___
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://play-lh.googleusercontent.com/51LDykvVt4B1EOfov5NmwGlHLbJ7kMd56kT7hcJb_-fUmgolJi8yJ4_mpaV8cijxSYw" width="300px" height="300px" />


# <font color= #bbc28d> **Cooking Mama 2: We're So Back** </font>
#### <font color= #2E9AFE> `Final Project: Text Generation`</font>
- <Strong> Sofía Maldonado, Diana Valdivia & Viviana Toledo </Strong>
- <Strong> Fecha </Strong>: 01/12/2025

<p style="text-align:right;"> Imagen recuperada de: https://play-lh.googleusercontent.com/51LDykvVt4B1EOfov5NmwGlHLbJ7kMd56kT7hcJb_-fUmgolJi8yJ4_mpaV8cijxSYw</p>

___

# <font color= #bbc28d> **Introduction** </font>

This project showcases a model that generates cooking recipes based on user input. For this, we are going to use the RecipeNLG dataset, created by researchers at the *Politechnika Poznańska* in Poland. This dataset contains over 2 million separate recipes for many different dishes from all the world's cuisines. 

Our aim with this project is to have a model that, given a specific list of ingredients, can give the user a recipe using those ingredients, with the recipe being generated in real time.

In [1]:
# Imports

# Generales
import pandas as pd
import numpy as np
import csv # <- Only used once
import random # <- Only used once
import math

# Transfomers
from transformers import pipeline, set_seed, GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
import accelerate
from datasets import Dataset

#Visualization
import gradio as gr

c:\Users\pixta\Documents\vscode\text-mining-projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


As mentioned above, the dataset has over 2 million recipes, but we are going to use a small subset: 20,000 recipes, or less than 1% of the original data. 

In [2]:
# SUBSET GENERATION, ONLY RUN ONCE
# Using Reservoir Sampling due to the size of the original
# random.seed(42)
# sample_size = 20000 

# reservoir = []

# with open('raw_data/full_dataset.csv', 'r', newline='', encoding='utf8') as f:
    # reader = csv.reader(f)
    # for i, row in enumerate(reader):
        # if i < sample_size:
            # reservoir.append(row)
        # else:
            # j = random.randrange(i + 1)
            # if j < sample_size:
                # reservoir[j] = row

# temp_data_storage = []
# for row in reservoir:
    # temp_data_storage.append(row)

# df = pd.DataFrame(temp_data_storage, columns=['Index','Dish','Ingredient_Amounts','Instructions','Source','Data_Source','Ingredients'])
# df.to_csv('processed_data/recipe_subset.csv')

The original dataset has 7 columns
| Column  | Description |
| ------------- |:-------------:|
| index      | definitely an index     |
| Dish      | The name of the dish     |
| Ingredient_Amounts      | The ingredients used in the recipe, with their respective amounts and units     |
| Instructions      | The steps to follow to cook the dish     |
| Source      | A URL with the source from which the recipe was scraped     |
| Data_Source      | Related with the source column, explained below     |
| Ingredients      | A list with just the ingredient names, with no quantities or units     |

The `Data_Source` column exists because some of the recipes were taken from another dataset called Recipe1M+, while others were scraped from the internet for this dataset. Since this information isn't really relevant for our purposes, we are going to drop it.

We are also going to drop the `Source` column for that same reason, as well as the `Ingredient_Amounts` column. This is because the instructions themselves (`Instructions`) already mention the quantities needed on each step, so ideally a user could just read the quantities there before cooking. 

# <font color= #bbc28d> **Pre-Processing** </font>

Before tokenization, we have to change the format of the text for the models to be able to grasp it.

Below an example of what this looks like, taking a random recipe from our subset

In [ ]:
s = "[""Wash and season chicken."", ""Roll chicken pieces in a plate of flour."", ""Put oil in frying pan under medium fire."", ""Put chicken in pan and brown.""]"

def first_step_preprocessing(text: str) -> str:
    processed_text = text.strip("[]")
    processed_text = text.replace('"', '')
    processed_text = text.replace("', '", ". ")
    processed_text = text.strip("[]") # No clue why this needs to happen, we ball tho

    if processed_text is None:
        return ""

    return processed_text

print(f"Example: {first_step_preprocessing(s)}")

Example: Wash and season chicken., Roll chicken pieces in a plate of flour., Put oil in frying pan under medium fire., Put chicken in pan and brown.


Now, we drop the unnecessary columns, and apply this `first_step_preprocessing` function to the `Instructions` and `Ingredients` columns

In [4]:
# First Step Preprocessing, also only run once!

to_change_columns = ['Ingredients', 'Instructions']

df = pd.read_csv('processed_data/recipe_subset.csv')
df[to_change_columns] = df[to_change_columns].map(first_step_preprocessing)
df = df.drop(columns=['Source','Data_Source','Ingredient_Amounts', 'Unnamed: 0'])
df.to_csv('processed_data/preprocessed_data.csv')

In [5]:
df_2 = pd.read_csv('processed_data/preprocessed_data.csv', keep_default_na=False) # <--- IMPORTANTISIMO
df_2

,Unnamed: 0,Index,Dish,Instructions,Ingredients
0,0,644563,Five Hour Beef Stew,"""Layer ingredients in Pyrex bowl and cover wit...","""stew meat"", ""carrots"", ""potatoes"", ""stalks ce..."
1,1,2067892,Pita Bread (Egyptian and Greek),"""Mix the ingredients with 3 deciliter (1 1/2 c...","""yeast"", ""salt"", ""olive oil"""
2,2,985324,Cranberry And Grand Marnier Relish,"""Chop cranberries in food processor and remove...","""fresh cranberries"", ""granny smith apples"", ""s..."
3,3,1832253,Fried Chicken And Blueberry Pancake Tacos,"""This is not the best fried chicken ever, but ...","""chicken tenders"", ""flour"", ""bread crumbs"", ""s..."
4,4,821664,Green Pepper Steak,"""Cut beef across grain into strips 1/8-inch th...","""beef chuck"", ""soy sauce"", ""clove garlic"", ""gi..."
...,...,...,...,...,...
19995,19995,19994,Dirt,"""You will also need 8 to 10 (7-ounce) plastic ...","""cold milk"", ""topping"", ""chocolate sandwich co..."
19996,19996,1303290,Chicken Divan,"""Preheat oven to 350. Place frozen broccoli in...","""chicken breasts"", ""frozen broccoli"", ""cream o..."
19997,19997,1570472,Hot Cross Scones,"""Preheat oven to 400\u00b0F. Line a baking she...","""All-purpose"", ""Whole Wheat Flour"", ""\u00bc"", ..."
19998,19998,182803,Gingerbread(One Serving),"""Stir and pour in kitchen Dixie cup."", ""Bake a...","""gingerbread mix"", ""water"""


Our next step is combining the "Ingredients", "Dish" and "Instructions" into a same column with input and output specifiers, for the model to be able to understand what our prompts are and what we are going to generate with them

In [14]:
df_2['Instructions'] = df_2['Instructions'].str.replace(r"\u00b0", "F", regex=False) # <- Fixing error with how the temperatures were shown
df_2['text'] = "Input: " + df_2['Ingredients'] + "\nOutput: " + df_2['Dish'] + ": " + df_2['Instructions']
print(df_2['text'][0]) # Example

Input: "stew meat", "carrots", "potatoes", "stalks celery", "green pepper", "onion", "tapioca", "salt", "sugar", "tomato juice"
Output: Five Hour Beef Stew: "Layer ingredients in Pyrex bowl and cover with aluminum foil. Bake 5 hours in 250F oven."


# <font color= #bbc28d> **Modeling** </font>

With the pre-processing complete, we can begin our modeling. We decided to use GPT-2, an older transformer that is very useful for general purpose tasks and one which we can easily fine-tune for our own purposes.

In [7]:
model_name = 'gpt2'

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

Now, we prepare the data to be loaded with hugging face

In [16]:
def tokenize(batch):
    tokens = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=256
    )
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

In [9]:
# Loading the dataset with Hugging Face
dataset = Dataset.from_pandas(df_2)
dataset = dataset.select_columns(['text'])

dataset = dataset.map(tokenize, batched=True)

dataset.set_format(type='torch', columns=['input_ids','attention_mask', 'labels'])
dataset

Map: 100%|██████████| 20000/20000 [00:21<00:00, 922.76 examples/s]


Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 20000
})

Now, we can begin training

In [15]:
# Training (ONLY RUN ONCE)

#args = TrainingArguments(
#    output_dir='./gpt2-ft',
#    per_device_eval_batch_size=256,
#    num_train_epochs=1,
#    fp16=True
#)

#trainer = Trainer(
#    model=model,
#    args=args,
#    train_dataset=dataset
#)

#trainer.train()
#model.save_pretrained('./gpt2-ft')
#tokenizer.save_pretrained('./gpt2-ft')

# <font color= #bbc28d> **Testing Model** </font>

With the model trained, we can quickly create a generator and test it out with some input

In [3]:
generator = pipeline('text-generation', model='./gpt2-ft', tokenizer='./gpt2-ft')

Device set to use cuda:0


In [6]:
prompt = 'Question: Cook something with paprika \nAnswer:'
print(generator(prompt, truncation=True))

[{'generated_text': 'Question: Cook something with paprika \nAnswer: "Cook paprika in an ovenproof skillet over medium heat.", "If you are baking an oven, let it burn for about 1 minute.", "Add paprika if desired.", "Pour paprika mixture into a shallow dish.", "Add a few handfuls of the ingredients.", "Cook on low for about 15 minutes or until golden brown and bubbly.", "Let it cook on high for about 15 minutes.", "I like to cook in the microwave for about 10 minutes.", "In a greased skillet over medium heat, cook the peppers until they turn golden and crisp.", "Let it cook for about 5 minutes."'}]


# <font color= #bbc28d> **Evaluation** </font>

We can also now check for eval metrics, specifically ***Perplexity***, which is the preferred choice when evaluating text generation tasks.

In [8]:
# Loading the model again

eval_model = GPT2LMHeadModel.from_pretrained("./gpt2-ft")
eval_tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-ft")
eval_tokenizer.pad_token = eval_tokenizer.eos_token

In [18]:
def eval_tokenize(batch):
    tokens = eval_tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=256
    )
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

In [19]:
# Getting Eval Dataset
eval_dataset = Dataset.from_pandas(df_2)
eval_dataset = eval_dataset.select_columns(['text'])

eval_dataset = eval_dataset.map(eval_tokenize, batched=True)

eval_dataset.set_format(type='torch', columns=['input_ids','attention_mask', 'labels'])
eval_dataset = eval_dataset.train_test_split(test_size=0.2)

Map: 100%|██████████| 20000/20000 [00:21<00:00, 944.36 examples/s]


In [1]:
# eval_args = TrainingArguments(
#     output_dir= "./gpt2-ft",
#     per_device_eval_batch_size=2
# )

# eval_trainer = Trainer(
#     model = eval_model,
#     args = eval_args,
#     eval_dataset= eval_dataset
# )

# metrics = eval_trainer.evaluate()

In [23]:
metrics

{'eval_train_loss': 1.137171983718872,
 'eval_train_model_preparation_time': 0.001,
 'eval_train_runtime': 324.2552,
 'eval_train_samples_per_second': 49.344,
 'eval_train_steps_per_second': 24.672,
 'eval_test_loss': 1.139114260673523,
 'eval_test_model_preparation_time': 0.001,
 'eval_test_runtime': 77.9668,
 'eval_test_samples_per_second': 51.304,
 'eval_test_steps_per_second': 25.652}

In [24]:
print(f"Perplexity: {math.exp(metrics['eval_test_loss'])}")

Perplexity: 3.124000089646566


The generations are fluid and easy to read, and seem kind of relevant to the task at hand (generating cooking recipes). However, they are not very logical. Sometimes some of the ingredients given are not mentioned again, and the steps taken for the cooking are non-sensical.

***ETHICAL CONSIDERATIONS***

This model can spit out cooking instructions that could be dangerous to try. For example, it can tell you to deep fry a frozen turkey, even though that would generate a massive fire which is obviously extremely dangerous.

**The instructions generated by this model should NOT be followed as they are. Alwasy make sure to double check.**

# <font color= #bbc28d> **Decoding Strategy Comparison** </font>

To compare different decoding strategies, we are going to use the same sample prompt with all of them.

In [2]:
comparison_prompt = 'Question: "meat" "cheese" "bread" \nAnswer: '

### <font color= #bbc28d> **Top-K Decoding** </font>

In [9]:
top_k_generator = pipeline('text-generation', model='./gpt2-ft', tokenizer='./gpt2-ft', top_k = 15, do_sample=True) # Default es 5
print(top_k_generator(comparison_prompt, truncation=True))

Device set to use cuda:0


[{'generated_text': 'Question: "meat" "cheese" "bread" \nAnswer:  : "Cook meat in hot oil. Remove from heat and let rest for a few minutes.", "Add cheese and cook for a few minutes.", "Add bread and bake until brown.", "Serve over rice."'}]


This configuration makes answers more verbose.

### <font color= #bbc28d> **Top-P Decoding** </font>

In [ ]:
top_p_generator = pipeline('text-generation', model='./gpt2-ft', tokenizer='./gpt2-ft', top_p = 0.5, do_sample=True) # Default es 1
print(top_p_generator(comparison_prompt, truncation=True)) 

Device set to use cuda:0


[{'generated_text': 'Question: "meat" "cheese" "bread" \nAnswer:  "Bread", "butter", "salt", "pepper", "eggs", "milk", "milk", "sour cream", "sugar", "butter", "butter", "sugar", "eggs", "butter", "butter", "salt", "pepper", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter", "butter'}]


This config makes answers look more like the original prompts.

### <font color= #bbc28d> **Temperature Decoding** </font>

In [19]:
temperature_generator = pipeline('text-generation', model='./gpt2-ft', tokenizer='./gpt2-ft', temperature = 0.7, do_sample=True)
print(temperature_generator(comparison_prompt, truncation=True)) 

Device set to use cuda:0


[{'generated_text': 'Question: "meat" "cheese" "bread" \nAnswer: ", "Cheddar cheese", "cornstarch", "sugar", "eggs", "vegetable oil", "water"\nOutput: Cheese Crock Pot: "Cook meat in fry pan.", "Add bread and stir until brown.", "Add cornstarch and sugar.", "Add oil and water and cook until thickened.", "Pour over meat."'}]


Bigger temperature ≈ more complex answers

# <font color= #bbc28d> **Showcase** </font>

Based on what we saw above, we are going to add a little bit of additional temperature to the generator used for our showcase.

In [7]:
gradio_generator = pipeline('text-generation', model='./gpt2-ft', tokenizer='./gpt2-ft', temperature = 0.7, do_sample = True, device='cuda:0')

Device set to use cuda:0


In [8]:
def create_gpt_prompt(user_prompt):
    final_prompt = "Question: " + user_prompt + "\nAnswer"

    result = gradio_generator(final_prompt, truncation=True)
    return result[0]['generated_text']

demo = gr.Interface(create_gpt_prompt, inputs=gr.Text(placeholder='Write a prompt'), outputs='text')

demo.launch()

Running on local URL:  http://127.0.0.1:7863

To create a public link, set `share=True` in `launch()`.


c:\Users\pixta\Documents\vscode\text-mining-projects\.venv\Lib\site-packages\gradio\analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


# <font color= #bbc28d> **References** </font>

- Bień, M., et al. (2020) **RecipeNLG: A Cooking Recipes Dataset for Semi-Structured Text Generation**. *Association for Computational Linguistics*. https://aclanthology.org/2020.inlg-1.4.pdf